In [ ]:
import numpy as np
import pandas as pd
import warnings
import pickle
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

# Install tensorflow if not already installed
# %pip install tensorflow
from tensorflow.keras.regularizers import l2
import tensorflow as tf
import tensorflow.keras.layers as tfl

import tensorflow_addons as tfa
from sklearn.utils import class_weight
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA

with open("features_data/diversity/train_dataset_1.pkl", 'rb') as file:
    positive_set = pickle.load(file)

with open("features_data/diversity/train_dataset_0.pkl", 'rb') as file:
    negative_set_entire = pickle.load(file)

column_names = ['pdb_name','residue','features','label']

# 确保 positive_set 和 negative_set_entire 是 DataFrame
if isinstance(positive_set, dict):
    positive_set = pd.DataFrame.from_dict(positive_set)
if isinstance(negative_set_entire, dict):
    negative_set_entire = pd.DataFrame.from_dict(negative_set_entire)

# ============================================================================
# SAMPLING RATIO PARAMETER (SRP) - Class Balancing Configuration
# ============================================================================
# ✅ USING PAPER'S OPTIMAL VALUE
# 
# Paper (Huang et al. 2025, Figure 2, page 6):
#   - Reports optimal SRP = 13 after testing range 5-15
#   - States: "An SRP of 13 yields the highest F1-score, indicating an 
#             optimal trade-off between precision and recall"
#
# Original Vendor Code:
#   - Used NEG_RATIO = 15 (equivalent to SRP = 15)
#   - Difference in performance is marginal (~2% in F1-score based on Figure 2)
#
# ✅ This implementation now uses NEG_RATIO = 13 to match paper exactly
# ============================================================================
NEG_RATIO = 13  # SRP = N(FRs) / N(AFRs) = negative-to-positive sampling ratio

print(f'✅ SRP Configuration:')
print(f'   Using SRP = {NEG_RATIO} (Paper optimal value)')
print(f'   Sampling {NEG_RATIO}:1 negative-to-positive ratio...\n')

# Randomly pick negative samples to balance with positive samples
Negative_Samples = negative_set_entire.sample(
    n=round(len(positive_set) * NEG_RATIO), 
    random_state=42
)

# combine positive and negative sets to make the final dataset
Train_set = pd.concat([positive_set, Negative_Samples], ignore_index=True, axis=0)

# collect the features and labels of train set
np.set_printoptions(suppress=True)
X_val = [0]*len(Train_set)
for i in range(len(Train_set)):
    feat = Train_set['features'][i]
    # 提取T5特征和bio特征
    # feat = np.concatenate((feat[:1024],feat[1044:]))
    X_val[i] = feat
X_train_orig = np.asarray(X_val)
y_val = Train_set['label'].to_numpy(dtype=float)
Y_train_orig = y_val.reshape(y_val.size,1)

# Generate a random order of elements with np.random.permutation and simply index into the arrays Feature and label 
idx = np.random.permutation(len(X_train_orig))
X_train,Y_train = X_train_orig[idx], Y_train_orig[idx]
scaler = StandardScaler()
scaler.fit(X_train) # fit on training set only
X_train = scaler.transform(X_train) # apply transform to the training set

# load test data
with open("features_data/diversity/test_dataset.pkl", 'rb') as file:
    Independent_test_set = pickle.load(file)

if isinstance(Independent_test_set, dict):
    Independent_test_set = pd.DataFrame.from_dict(Independent_test_set)
# collect the features and labels for independent set
X_independent = [0]*len(Independent_test_set)
for i in range(len(Independent_test_set)):
    feat1 = Independent_test_set['features'][i]
    # feat1 = Independent_test_set['features'][i]
    # feat1 = np.concatenate((feat1[:1024],feat1[1044:]))
    X_independent[i] = feat1
X_test = np.asarray(X_independent)
y_independent = Independent_test_set['label'].to_numpy(dtype=float)
Y_test = y_independent.reshape(y_independent.size,1)
X_test = scaler.transform(X_test) # apply standardization (transform) to the test set


In [ ]:
# ============================================================================
# OPTIONAL: TOPOLOGY AUGMENTATION (DCI + Betweenness)
# ============================================================================
# Appends topological features to baseline features (1047D → 1050D)
#
# Features Added:
#   - DCI (Deformation-Curvature Index): 2D vector from persistent homology
#   - Betweenness Centrality: 1D scalar from residue contact network
#
# Requirements:
#   - PDB files in 'Case Study/' or 'AlloFusion-main/Case Study/'
#   - Dependencies: prody, networkx
#
# Toggle: Set USE_DCI_BETWEENNESS = True to enable
# ============================================================================

from pathlib import Path
import numpy as np
from sklearn.preprocessing import StandardScaler

# Toggle topology augmentation
USE_DCI_BETWEENNESS = False  # Set to True to enable topology features

PDB_CACHE_DIR = Path('AlloFusion-main/Case Study') if Path('AlloFusion-main/Case Study').exists() else Path('Case Study')
CUTOFF = 8.0
SIGMA = 4.0

def augment_df_with_topology(df, pdb_cache_dir):
    """
    Augment residue features with topological descriptors (DCI + Betweenness).
    
    For each residue, appends:
        - DCI (Deformation-Curvature Index): 2D vector from persistent homology
        - Betweenness Centrality: 1D scalar from residue contact network
    
    Total augmentation: +3 features per residue (1047 → 1050 dimensions)
    
    Args:
        df: DataFrame with columns ['pdb_name', 'features', 'label']
            - pdb_name format: "{PDB_ID}_{CHAIN_ID}" (e.g., "4ZSI_B")
            - features: numpy array of shape (1047,)
        pdb_cache_dir: Path to directory containing PDB files
    
    Returns:
        DataFrame with augmented features (shape: 1050,)
    
    Notes:
        - Requires PDB structure files in pdb_cache_dir
        - Missing structures → zero-padded features (graceful degradation)
        - Dependencies: prody, networkx, tda_adapter, utils.graph_features
    """
    try:
        from tda_adapter.dci import compute_and_cache_dci
        from utils.graph_features import compute_betweenness_centrality
    except Exception as e:
        print(f"[Topo] Missing dependencies: {e}. Install 'prody' and 'networkx'. Skipping augmentation.")
        return df
    
    out = df.copy()
    extras = {}
    
    # Precompute per-protein extras
    for name, grp in out.groupby('pdb_name', sort=False):
        nm = str(name).strip('>')
        parts = nm.split('_')
        L = len(grp)
        dci = np.zeros((L, 2), dtype=float)
        btw = np.zeros(L, dtype=float)
        
        if len(parts) == 2:
            pdb_id, chain_id = parts
            pdb_path = Path(pdb_cache_dir) / f"{pdb_id}.pdb"
            try:
                if pdb_path.exists():
                    # Compute DCI (Deformation-Curvature Index)
                    dci_raw = compute_and_cache_dci(
                        str(pdb_path), chain_id, 
                        cache_dir=str(pdb_cache_dir), 
                        cutoff=CUTOFF
                    )
                    m = min(L, dci_raw.shape[0]) if dci_raw is not None else 0
                    if m > 0:
                        dci[:m] = dci_raw[:m]
                    
                    # Compute betweenness centrality
                    btw_raw = compute_betweenness_centrality(
                        str(pdb_path), chain_id, 
                        cutoff=CUTOFF, sigma=SIGMA, 
                        normalize=True
                    )
                    m2 = min(L, btw_raw.shape[0]) if btw_raw is not None else 0
                    if m2 > 0:
                        btw[:m2] = btw_raw[:m2]
            except Exception as e:
                print(f"[Topo] {nm}: failed ({e}); using zeros")
        else:
            print(f"[Topo] Unrecognized pdb_name format: {name}; using zeros")
        
        extras[name] = (dci, btw)
    
    # Append extras row-wise (assumes group order follows residue order)
    counters = {}
    new_feats = []
    for idx, row in out.iterrows():
        name = row['pdb_name']
        pos = counters.get(name, 0)
        counters[name] = pos + 1
        dci, btw = extras.get(name, (None, None))
        base = row['features']
        if dci is not None and btw is not None and pos < len(btw):
            base = np.concatenate([base, dci[pos], [btw[pos]]])
        new_feats.append(base)
    out['features'] = new_feats
    return out

# Apply topology augmentation if enabled
if USE_DCI_BETWEENNESS:
    print('\n' + '='*70)
    print('TOPOLOGY AUGMENTATION: DCI + Betweenness')
    print('='*70)
    print('Augmenting training and test sets with topological features...')
    
    # Augment training set (needs to be reconstructed from positive_set and Negative_Samples)
    train_df = pd.concat([positive_set, Negative_Samples], ignore_index=True, axis=0)
    train_df_augmented = augment_df_with_topology(train_df, PDB_CACHE_DIR)
    
    # Augment test set
    test_df_augmented = augment_df_with_topology(Independent_test_set, PDB_CACHE_DIR)
    
    # Rebuild X_train with augmented features
    X_val_augmented = [0]*len(train_df_augmented)
    for i in range(len(train_df_augmented)):
        X_val_augmented[i] = train_df_augmented['features'][i]
    X_train_orig = np.asarray(X_val_augmented)
    y_val = train_df_augmented['label'].to_numpy(dtype=float)
    Y_train_orig = y_val.reshape(y_val.size, 1)
    
    # Shuffle
    idx = np.random.permutation(len(X_train_orig))
    X_train, Y_train = X_train_orig[idx], Y_train_orig[idx]
    
    # Re-standardize with augmented features
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    
    # Rebuild X_test with augmented features
    X_independent_augmented = [0]*len(test_df_augmented)
    for i in range(len(test_df_augmented)):
        X_independent_augmented[i] = test_df_augmented['features'][i]
    X_test = np.asarray(X_independent_augmented)
    y_independent = test_df_augmented['label'].to_numpy(dtype=float)
    Y_test = y_independent.reshape(y_independent.size, 1)
    X_test = scaler.transform(X_test)
    
    feat_dim = X_train.shape[1]
    print(f'\n✓ Augmentation complete!')
    print(f'  Feature dimension: {feat_dim} (baseline: 1047, augmented: 1050)')
    print(f'  Training samples: {X_train.shape[0]:,}')
    print(f'  Test samples: {X_test.shape[0]:,}')
    print('='*70 + '\n')
else:
    print('\n[Topo] USE_DCI_BETWEENNESS=False; using baseline features (1047D)')
    print('       To enable topology features, set USE_DCI_BETWEENNESS=True\n')


In [ ]:
import tensorflow as tf
import tensorflow.keras.layers as tfl
from kerastuner import HyperModel
from kerastuner.tuners import BayesianOptimization
from kerastuner import Objective
import keras_tuner

feat_shape = X_train[0].size
# 定义CNN模型，接收超参数
class CNNHyperModel(HyperModel):
    def build(self, hp):
        model = tf.keras.Sequential()
        
        # 第一层卷积层
        model.add(tfl.Conv1D(
            filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32), 
            kernel_size=hp.Int('conv1_kernel_size', min_value=3, max_value=7, step=2),
            activation='relu',
            input_shape=(feat_shape, 1)
        ))
        model.add(tfl.BatchNormalization())
        model.add(tfl.Dropout(rate=hp.Float('dropout1_rate', min_value=0.2, max_value=0.5, step=0.1)))

        # 第二层卷积层
        model.add(tfl.Conv1D(
            filters=hp.Int('conv2_filters', min_value=64, max_value=256, step=64),
            kernel_size=hp.Int('conv2_kernel_size', min_value=3, max_value=7, step=2),
            activation='relu'
        ))
        model.add(tfl.BatchNormalization())
        model.add(tfl.Dropout(rate=hp.Float('dropout2_rate', min_value=0.2, max_value=0.5, step=0.1)))

        # 第三层卷积层
        model.add(tfl.Conv1D(
            filters=hp.Int('conv3_filters', min_value=32, max_value=128, step=32),
            kernel_size=hp.Int('conv3_kernel_size', min_value=3, max_value=7, step=2),
            activation='relu'
        ))
        model.add(tfl.BatchNormalization())
        model.add(tfl.Dropout(rate=hp.Float('dropout3_rate', min_value=0.2, max_value=0.5, step=0.1)))

         # 新增的第四层卷积层
        model.add(tfl.Conv1D(
            filters=hp.Int('conv4_filters', min_value=32, max_value=128, step=32),
            kernel_size=hp.Int('conv4_kernel_size', min_value=3, max_value=7, step=2),
            activation='relu'
        ))
        model.add(tfl.BatchNormalization())
        model.add(tfl.Dropout(rate=hp.Float('dropout4_rate', min_value=0.2, max_value=0.5, step=0.1)))

        # Flatten层
        model.add(tfl.Flatten())

        # 全连接层
        model.add(tfl.Dense(
            units=hp.Int('dense_units', min_value=64, max_value=256, step=64), 
            activation='relu'
        ))

        # 输出层
        model.add(tfl.Dense(1, activation='sigmoid'))

        # 编译模型
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='LOG')
            ),
            loss='binary_crossentropy',
            metrics=['AUC']
        )

        return model


# 选择贝叶斯优化调优器
tuner = BayesianOptimization(
    CNNHyperModel(),
    objective=Objective("val_auc", direction="max"),  # 优化目标
    max_trials=10,  # 最大试验次数
    executions_per_trial=1,  # 每个试验执行一次
    directory='keras_tuner_dir',  # 存储日志的目录
    project_name='cnn_hyperparam_tuning4'  # 项目名称
)

# 调整模型的超参数
tuner.search(
    X_train, Y_train,  # 训练数据
    epochs=10,  # 训练轮数
    validation_data=(X_test, Y_test),  # 验证数据
    batch_size=32  # 批大小
)

# 获取最佳超参数组合
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:", best_hyperparameters.values)

# 使用最佳超参数训练最终模型
best_model = tuner.hypermodel.build(best_hyperparameters)
best_model.fit(X_train,Y_train, epochs=10, validation_data=(X_test, Y_test))


In [ ]:
def CNN_Model():
    
    model = tf.keras.Sequential()
    model.add(tfl.Conv1D(32, 3, padding='same', activation='relu', input_shape=(feat_shape,1)))
    model.add(tfl.BatchNormalization())
    model.add(tfl.Dropout(0.2)) # 0.23

    model.add(tfl.Conv1D(128, 3, padding='same',activation='relu'))
    model.add(tfl.BatchNormalization())
    model.add(tfl.Dropout(0.3)) # 0.21

    model.add(tfl.Conv1D(32, 5, padding='same',activation='relu'))
    model.add(tfl.BatchNormalization()) 
    model.add(tfl.Dropout(0.2)) # 0.47

    model.add(tfl.Conv1D(32, 3, padding='same',activation='relu'))
    model.add(tfl.BatchNormalization()) 
    model.add(tfl.Dropout(0.3)) # 0.47

    model.add(tfl.Flatten())
    model.add(tfl.Dense(128, activation='relu'))
    # model.add(tfl.Dropout(0.5))

    model.add(tfl.Dense(32, activation='relu'))
    model.add(tfl.Dense(1, activation='sigmoid'))
    
    return model

#{'conv1_filters': 32, 'conv1_kernel_size': 3, 'dropout1_rate': 0.4, 'conv2_filters': 192, 'conv2_kernel_size': 5, 'dropout2_rate': 0.2, 'conv3_filters': 128, 'conv3_kernel_size': 5, 'dropout3_rate': 0.30000000000000004, 'dense_units': 128, 'learning_rate': 0.00015872369686433261
# {'conv1_filters': 32, 'conv1_kernel_size': 3, 'dropout1_rate': 0.2, 'conv2_filters': 128, 'conv2_kernel_size': 3, 'dropout2_rate': 0.30000000000000004, 'conv3_filters': 32, 'conv3_kernel_size': 5, 'dropout3_rate': 0.2, 'conv4_filters': 32, 'conv4_kernel_size': 3, 'dropout4_rate': 0.30000000000000004, 'dense_units': 128, 'learning_rate': 0.000735323218543868}


feat_shape = X_train[0].size
# print("feat_shape",feat_shape)
cnn_model = CNN_Model()

learning_rate = 0.0001
optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate)
cnn_model.compile(optimizer=optimizer,
                    loss='binary_crossentropy',
                    metrics=['AUC', 'accuracy', 'Precision', 'Recall'])

cnn_model.summary()

# ============================================================================
# STRATIFIED VALIDATION SPLIT (Imbalanced Data Best Practice)
# ============================================================================
# Paper mentions 80/20 train/val split but doesn't specify stratification method.
# For imbalanced datasets (AFR:FR ≈ 1:15), stratified splitting ensures:
#   - Validation set maintains same class distribution as training set
#   - More reliable validation metrics
#   - Consistent performance estimation
#
# Implementation: Use sklearn.model_selection.train_test_split with stratify=Y
# ============================================================================
from sklearn.model_selection import train_test_split

# Train the Model
batch_size = 32 # 32
epochs = 100
VAL_SPLIT = 0.2

print('\n' + '='*70)
print('STRATIFIED TRAIN/VALIDATION SPLIT')
print('='*70)
print(f'Using stratified splitting to maintain class balance')
print(f'Validation split: {int(VAL_SPLIT*100)}%\n')

# Create stratified train/validation split
X_train_final, X_val_final, Y_train_final, Y_val_final = train_test_split(
    X_train, Y_train,
    test_size=VAL_SPLIT,
    stratify=Y_train,  # Ensures same AFR:FR ratio in both sets
    random_state=42
)

print(f'Training samples:   {len(X_train_final):,}')
print(f'  AFRs: {int(Y_train_final.sum())} ({100*Y_train_final.mean():.1f}%)')
print(f'  FRs:  {len(Y_train_final) - int(Y_train_final.sum())} ({100*(1-Y_train_final.mean()):.1f}%)')
print(f'\nValidation samples: {len(X_val_final):,}')
print(f'  AFRs: {int(Y_val_final.sum())} ({100*Y_val_final.mean():.1f}%)')
print(f'  FRs:  {len(Y_val_final) - int(Y_val_final.sum())} ({100*(1-Y_val_final.mean()):.1f}%)')
print('='*70 + '\n')

# 学习率调度器： ReduceLROnPlateau
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc', 
    factor=0.5, 
    patience=3, 
    min_lr=1e-5, 
    verbose=1)
checkpoint = tf.keras.callbacks.ModelCheckpoint("myModel/multy1/embedding-pssm-bio15.h5", save_best_only=True) # save the best model weights 仅保存验证集上性能最好的最佳模型权重
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True,verbose=1) # stop training if the validation AUC does not improve for 3 epochs

# Train with explicit stratified validation data (not validation_split parameter)
history = cnn_model.fit(
    X_train_final, Y_train_final, 
    epochs=epochs, 
    batch_size=batch_size, 
    validation_data=(X_val_final, Y_val_final),  # ✓ Stratified validation set
    callbacks=[checkpoint, early_stopping, lr_scheduler]
)

df_loss_auc = pd.DataFrame(history.history)

# 创建副本并重命名列
df_loss= df_loss_auc[['loss','val_loss']].copy()
df_loss.rename(columns={'loss':'train','val_loss':'validation'},inplace=True)
 
df_auc= df_loss_auc[['auc','val_auc']].copy()
df_auc.rename(columns={'auc':'train','val_auc':'validation'},inplace=True)

# 绘制损失和 AUC 曲线
Model_Loss_plot_title = 'Model Loss'
df_loss.plot(title=Model_Loss_plot_title,figsize=(12,8)).set(xlabel='Epoch',ylabel='Loss')

Model_AUC_plot_title = 'Model AUC'
df_auc.plot(title=Model_AUC_plot_title,grid=True,figsize=(12,8)).set(xlabel='Epoch',ylabel='AUC')

# 绘制accuracy, precision, recall
df_accuracy = pd.DataFrame(history.history)
df_accuracy[['accuracy','val_accuracy']].plot(title='Model Accuracy',grid=True,figsize=(12,8)).set(xlabel='Epoch',ylabel='Accuracy')
df_precision = pd.DataFrame(history.history)
df_precision[['precision','val_precision']].plot(title='Model Precision',grid=True,figsize=(12,8)).set(xlabel='Epoch',ylabel='Precision')
df_recall = pd.DataFrame(history.history)
df_recall[['recall','val_recall']].plot(title='Model Recall',grid=True,figsize=(12,8)).set(xlabel='Epoch',ylabel='Recall')


eval_result = cnn_model.evaluate(X_test, Y_test)
print(f"test loss: {round(eval_result[0],4)}, test auc: {round(eval_result[1],4)}, test accuracy: {round(eval_result[2],4)}, test precision: {round(eval_result[3],4)}, test recall: {round(eval_result[4],4),}")
